# JSON schema mode
Native structured output: the response is constrained to a JSON schema generated from a Pydantic model.

In [1]:
import json
import time

from dotenv import load_dotenv
from google import genai
from google.genai import types
from IPython.display import JSON, Markdown, display

load_dotenv()  # loads GEMINI_API_KEY from .env
client = genai.Client()

from demo_helpers import VALID_APPLICATIONS

In [2]:
transcript = "Hey Google, navigate to Ara."

In [9]:
from pydantic import BaseModel, Field


class ParsedCommand(BaseModel):
    target_application: str = Field(description="The application the user wants to use")
    target_action: str = Field(
        description="The action to perform within that application"
    )
    additional_parameters: dict[str, str] = Field(
        default_factory=dict,
        description=(
            "Parameters for the action mentioned in the transcript, as a mapping "
            "of parameter name (as listed for the chosen action) to value"
        ),
    )


print(json.dumps(ParsedCommand.model_json_schema(),indent=2))

{
  "properties": {
    "target_application": {
      "description": "The application the user wants to use",
      "title": "Target Application",
      "type": "string"
    },
    "target_action": {
      "description": "The action to perform within that application",
      "title": "Target Action",
      "type": "string"
    },
    "additional_parameters": {
      "additionalProperties": {
        "type": "string"
      },
      "description": "Parameters for the action mentioned in the transcript, as a mapping of parameter name (as listed for the chosen action) to value",
      "title": "Additional Parameters",
      "type": "object"
    }
  },
  "required": [
    "target_application",
    "target_action"
  ],
  "title": "ParsedCommand",
  "type": "object"
}


## Input

In [10]:
message = f"""For the following transcript, determine the target action the user wants to take,
the target application, and any additional parameters.\n
The target application, action and parameters must be chosen from the following
valid applications (each lists its actions and the parameters each action accepts):\n
{json.dumps(VALID_APPLICATIONS, indent=2)}\n

transcript: {transcript}"""
print(message)

For the following transcript, determine the target action the user wants to take,
the target application, and any additional parameters.

The target application, action and parameters must be chosen from the following
valid applications (each lists its actions and the parameters each action accepts):

{
  "maps": {
    "description": "Navigation and location search",
    "actions": {
      "navigate": {
        "destination": "str",
        "mode": "driving | walking | cycling | transit"
      },
      "search_nearby": {
        "query": "str",
        "radius_km": "float"
      },
      "show_traffic": {}
    }
  },
  "music": {
    "description": "Music streaming and playback",
    "actions": {
      "play": {
        "song": "str",
        "artist": "str",
        "playlist": "str"
      },
      "pause": {},
      "skip": {},
      "set_volume": {
        "level": "int (0-100)"
      }
    }
  },
  "messages": {
    "description": "Send and read text messages",
    "actions": {
   

In [11]:
config = types.GenerateContentConfig(
    response_mime_type="application/json",
    response_json_schema=ParsedCommand.model_json_schema(),
)
config

GenerateContentConfig(
  response_json_schema={
    'properties': {
      'additional_parameters': {
        'additionalProperties': {
          'type': 'string'
        },
        'description': 'Parameters for the action mentioned in the transcript, as a mapping of parameter name (as listed for the chosen action) to value',
        'title': 'Additional Parameters',
        'type': 'object'
      },
      'target_action': {
        'description': 'The action to perform within that application',
        'title': 'Target Action',
        'type': 'string'
      },
      'target_application': {
        'description': 'The application the user wants to use',
        'title': 'Target Application',
        'type': 'string'
      }
    },
    'required': [
      'target_application',
      'target_action',
    ],
    'title': 'ParsedCommand',
    'type': 'object'
  },
  response_mime_type='application/json'
)

## Model call

In [13]:
start = time.perf_counter()
response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=message,
    config=config,
)
print(f"Client call took {time.perf_counter() - start:.2f}s")

Client call took 1.06s


## Output

In [15]:
print(response)

sdk_http_response=HttpResponse(
  headers=<dict len=12>
) candidates=[Candidate(
  content=Content(
    parts=[
      Part(
        text="""{
  "target_application": "maps",
  "target_action": "navigate",
  "additional_parameters": {
    "destination": "Ara"
  }
}""",
        thought_signature=b'\x12`\n^\x01i\x14}\x13\xd2N*\xdd\xbf\xc6.\xe1A\xf4`\xf3_\x9c\xa4\x828\x9b\xa4&\xbc\xbfL\xec\x86\xb7\x80Y\xf7%\x8b\x06\xefp\xaa(8\xe4\x17\xc5\x8c\xc4p\xb7\x1c\xd9\x87\xca\xdc\x91\x13\x1c*\xca\x8f\xaaq\x16\x9c\xe3%_\x88\xcc\t\x8a\xf2\x81t\xdf\x11\xb3\xb7\xa7p\xb8\x9c\x8es\x03\xb5]\xe4%m\x08\xea\xe0\x94'
      ),
    ],
    role='model'
  ),
  finish_reason=<FinishReason.STOP: 'STOP'>,
  index=0
)] create_time=None model_version='gemini-3.5-flash-lite' prompt_feedback=None response_id='c8r3avjmAsGF694PtIWD8Qc' usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=42,
  prompt_token_count=546,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TE

In [16]:
# response.parsed is the already-decoded JSON (a dict); validate it into the Pydantic model
result = ParsedCommand.model_validate(response.parsed)
result

ParsedCommand(target_application='maps', target_action='navigate', additional_parameters={'destination': 'Ara'})